# Finetuning using a domain split

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
import sys
from pathlib import Path

PROJECT_ROOT = next(
    root
    for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "components").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from components.config import (
    DATA_DIR, LABEL_NAMES, MAX_LEN, PROJECT_ROOT, RAW_DATA_PATH, SEED, WEIGHTS_PATH,
)
from components.data import (
    balanced_subset, domain_overlap, load_urls, make_loaders, save_splits, split_dataset,
)
from components.evaluation import compute_metrics, misclassified
from components.inference import (
    build_classifier, classify_url_name, get_tokenizer, save_classifier,
)
from components.metrics import calc_accuracy_loader, calc_loss_loader, predict_proba_loader
from components.preprocessing import normalize_url
from components.training import build_optimizer, train_classifier

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = get_tokenizer()
print(f"Device: {device}")

## Data

In [ ]:
data = load_urls(RAW_DATA_PATH) 
print(len(data), "unique URLs")
print(data["label"].value_counts())  

In [ ]:
N_SAMPLES = 20_000
SPLIT_BY = "domain" 

subset_df = balanced_subset(data, N_SAMPLES)
train_df, val_df, test_df = split_dataset(subset_df, group_by_domain=(SPLIT_BY == "domain"))

print(f"{len(train_df)} train / {len(val_df)} val / {len(test_df)} test")
print(
    f"Phishing share: train {1 - train_df['label'].mean():.2f}, "
    f"val {1 - val_df['label'].mean():.2f}, test {1 - test_df['label'].mean():.2f}"
)
print(
    f"URLs whose domain also appears in train: "
    f"val {domain_overlap(train_df, val_df):.1%}, test {domain_overlap(train_df, test_df):.1%}"
)

save_splits(train_df, val_df, test_df, DATA_DIR)

In [ ]:
for url in ["google.com", "http://google.com", "https://google.com", "https://www.google.com"]:
    print(f"{url:<26} -> {normalize_url(url)}")

In [ ]:
BATCH_SIZE = 32
train_loader, val_loader, test_loader = make_loaders(
    DATA_DIR, tokenizer, MAX_LEN, batch_size=BATCH_SIZE
)

input_batch, target_batch = next(iter(train_loader))
print("Input batch dimensions:", input_batch.shape) 
print("Label batch dimensions:", target_batch.shape)
print(f"{len(train_loader)} training / {len(val_loader)} validation / {len(test_loader)} test batches")

## Model

In [ ]:
model = build_classifier(pretrained=True, models_dir=str(PROJECT_ROOT / "model/gpt2"))
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(model.out_head)
print(f"Trainable parameters: {trainable:,} of {total:,}")

In [ ]:
with torch.no_grad():
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
print(f"Before training: val accuracy {val_accuracy*100:.2f}%, val loss {val_loss:.3f}")

## Training

In [ ]:
NUM_EPOCHS = 2
EVAL_FREQ = 100
EVAL_ITER = 10

optimizer = build_optimizer(model)  

start = time.perf_counter()
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=NUM_EPOCHS, eval_freq=EVAL_FREQ, eval_iter=EVAL_ITER,
)
print(f"Training completed in {(time.perf_counter() - start) / 60:.2f} minutes.")

In [ ]:
import matplotlib.pyplot as plt

steps = [i * EVAL_FREQ for i in range(len(train_losses))]
plt.plot(steps, train_losses, label="train")
plt.plot(steps, val_losses, label="validation")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
save_classifier(model)
print(f"Saved to {WEIGHTS_PATH}")

## Evaluation

In [ ]:
test_csv = pd.read_csv(DATA_DIR / "test.csv")  
p_phish, labels = predict_proba_loader(test_loader, model, device)
preds = np.where(p_phish > 0.5, 0, 1)

print(f"Test accuracy: {(preds == labels).mean()*100:.2f}% ({len(labels)} URLs)")
print(classification_report(
    labels, preds,
    target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
    digits=3,
))
print(confusion_matrix(labels, preds)) 

In [ ]:
metrics = compute_metrics(p_phish, labels)
for name, value in metrics.items():
    print(f"{name:<24}{value:.4f}")

In [ ]:
print(f"{'threshold':>9} {'recall':>8} {'FPR':>8}")
for threshold in (0.1, 0.3, 0.5, 0.7, 0.9):
    m = compute_metrics(p_phish, labels, threshold=threshold)
    print(f"{threshold:>9.1f} {m['recall_phishing']:>8.4f} {m['false_positive_rate']:>8.4f}")

In [ ]:
errors = misclassified(test_csv, p_phish)
print(f"{len(errors)} misclassified of {len(test_csv)}")
errors.sort_values("p_phish")  

In [ ]:
for url in [
    "google.com",
    "http://google.com",
    "https://google.com",
    "https://www.google.com",
    "https://www.paypa1-account-verify.com/login",
]:
    label, confidence = classify_url_name(url, model, tokenizer, device)
    print(f"{url:<46} {label} ({confidence:.1%})")